# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

### Google Colab

Install dependencies into the **current** Colab runtime (no `uv` or extra kernel). Use a **GPU** runtime for vLLM. After this finishes, use **Runtime → Restart runtime** once, then continue.


In [ ]:
# Colab: skip the `uv` cell above when using this path.
#
# Plain `pip install vllm` often pulls a CUDA 13-built wheel (needs libcudart.so.13), which Colab does not provide on LD_LIBRARY_PATH.
# Use PyTorch cu129 + the official vLLM +cu129 release wheel so everything targets CUDA 12.9 (matches Colab drivers).
# GPU VMs are usually x86_64. On aarch64, replace `x86_64` with `aarch64` in the wheel URL.
#
%pip install -q sympy numpy tqdm "antlr4-python3-runtime==4.11.1"
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu129 --upgrade
%pip install -q "https://github.com/vllm-project/vllm/releases/download/v0.20.0/vllm-0.20.0+cu129-cp38-abi3-manylinux_2_31_x86_64.whl" --extra-index-url https://download.pytorch.org/whl/cu129
%pip install -q transformers "bitsandbytes>=0.48.1"


## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — results filename (default `results/starter_results.jsonl`). On **Colab**, after the Drive setup cell, saves go to **`My Drive/CSE151B/results/<filename>`**; locally, relative to the working directory
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [1]:
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # Colab / single-GPU: must be "0". Use "1" only if a 2nd GPU exists.
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/full_public_8k.jsonl"
MAX_TOKENS  = 8192                   # was 32768 — thinking models generate massive CoT, cap saves huge time

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
# V1 default starts a separate EngineCore process; that commonly fails in Jupyter/Colab.
# In-process mode avoids that (see vLLM LLMEngine.from_engine_args + envs.VLLM_ENABLE_V1_MULTIPROCESSING).
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

import glob
import site

def _prepend_nvidia_libs_to_ld_path() -> None:
    roots = list(site.getsitepackages())
    user_site = getattr(site, "getusersitepackages", lambda: None)()
    if user_site:
        roots.append(user_site)
    libdirs: list[str] = []
    seen: set[str] = set()
    for root in roots:
        for d in glob.glob(os.path.join(root, "nvidia", "*", "lib")):
            if os.path.isdir(d) and d not in seen:
                seen.add(d)
                libdirs.append(d)
    if libdirs:
        sep = os.pathsep
        os.environ["LD_LIBRARY_PATH"] = sep.join(libdirs + [os.environ.get("LD_LIBRARY_PATH", "")]).strip(sep)


_prepend_nvidia_libs_to_ld_path()

import contextlib
import io
import re
import sys
from pathlib import Path
from typing import Optional

@contextlib.contextmanager
def _jupyter_stdout_for_vllm():
    """Temporarily give vLLM a real stdout FD; restores notebook stdout after."""
    try:
        sys.stdout.fileno()
    except (io.UnsupportedOperation, AttributeError, OSError):
        saved_out, saved_err = sys.stdout, sys.stderr
        dup1, dup2 = os.dup(1), os.dup(2)
        try:
            sys.stdout = os.fdopen(dup1, "w", buffering=1)
            sys.stderr = os.fdopen(dup2, "w", buffering=1)
            yield
        finally:
            sys.stdout.close()
            sys.stderr.close()
            sys.stdout, sys.stderr = saved_out, saved_err
    else:
        yield

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

### Colab: put `public.jsonl` on Google Drive, then copy it here

The Colab VM does not include your git repo. Upload **`public.jsonl`** to  
`My Drive/CSE151B/data/public.jsonl` (or edit `DRIVE_JSONL` in the next cell), run the next cell, then load the dataset.


In [2]:
# Google Colab — copy `public.jsonl` from Drive into `data/` (runtime has no repo files until you do).
# One-time: in Drive, create `CSE151B/data/` and upload `public.jsonl` there (from the assignment zip/repo).
# Change `DRIVE_JSONL` if you store the file elsewhere.

import shutil
from pathlib import Path

DRIVE_JSONL = Path("/content/drive/MyDrive/CSE151B/data/public.jsonl")
# Same assignment folder as data — put judger.py here for scoring.
DRIVE_PROJECT_ROOT = DRIVE_JSONL.parent.parent

try:
    from google.colab import drive
except ImportError:
    print("Skip: not Google Colab — use `data/public.jsonl` from your local repo clone.")
else:
    drive.mount("/content/drive")
    if not DRIVE_JSONL.is_file():
        raise FileNotFoundError(
            f"Missing on Drive: {DRIVE_JSONL}\n"
            "Upload public.jsonl to that path or set DRIVE_JSONL to your file."
        )
    Path("data").mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE_JSONL, "data/public.jsonl")
    print(f"Copied to {Path('data/public.jsonl').resolve()}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copied to /content/data/public.jsonl


In [3]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [4]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. ...

── Free-form user prompt (first 200 chars) ──
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS] ...



## 5. Load Model with vLLM

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

with _jupyter_stdout_for_vllm():
    llm = LLM(
        model=MODEL_ID,
        quantization="bitsandbytes",
        load_format="bitsandbytes",
        enable_prefix_caching=True,      # was False — reuses shared system prompt prefix across questions
        # enforce_eager removed           # was True — disabling CUDA graphs was a major bottleneck
        gpu_memory_utilization=0.88,
        max_model_len=16384,
        trust_remote_code=True,
        max_num_seqs=64,
        max_num_batched_tokens=16384,    # was 8192 — raised to match max_model_len for better throughput
    )

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

INFO 05-05 00:17:55 [core.py:109] Initializing a V1 LLM engine (v0.20.0) with config: model='Qwen/Qwen3-4B-Thinking-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Thinking-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=16384, download_dir=None, load_format=bitsandbytes, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=bitsandbytes, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_

model.safetensors.index.json: 0.00B [00:00, ?B/s]

INFO 05-05 00:18:21 [weight_utils.py:615] Time spent downloading weights for Qwen/Qwen3-4B-Thinking-2507: 20.705110 seconds
INFO 05-05 00:18:22 [weight_utils.py:904] Filesystem type for checkpoints: OVERLAY. Checkpoint size: 7.49 GiB. Available RAM: 79.97 GiB.
INFO 05-05 00:18:22 [weight_utils.py:927] Auto-prefetch is disabled because the filesystem (OVERLAY) is not a recognized network FS (NFS/Lustre). If you want to force prefetching, start vLLM with --safetensors-load-strategy=prefetch.


Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 05-05 00:18:25 [gpu_model_runner.py:4879] Model loading took 2.7 GiB memory and 28.053071 seconds
INFO 05-05 00:18:37 [backends.py:1069] Using cache directory: /root/.cache/vllm/torch_compile_cache/95bbff7442/rank_0_0/backbone for vLLM's torch.compile
INFO 05-05 00:18:37 [backends.py:1128] Dynamo bytecode transform time: 10.96 s
INFO 05-05 00:18:47 [backends.py:376] Cache the graph of compile range (1, 16384) for later use
INFO 05-05 00:18:54 [backends.py:391] Compiling a graph for compile range (1, 16384) takes 16.28 s
INFO 05-05 00:19:01 [decorators.py:668] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/364deebc07fb29298a030d0ccc2e51dc97cb384b3e76521da7ac217f400d3e58/rank_0_0/model
INFO 05-05 00:19:01 [monitor.py:53] torch.compile took 35.03 s in total
INFO 05-05 00:19:01 [monitor.py:81] Initial profiling/warmup run took 0.25 s
INFO 05-05 00:19:11 [gpu_model_runner.py:5963] Profiling CUDA graph memory: PIECEWISE=19 (largest=128), FULL=11 (

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

In [6]:
CHUNK_SIZE = 100  # checkpoint every N questions

# Mirror the same Drive-aware path logic used by the save cell
try:
    _results_dir = DRIVE_PROJECT_ROOT / "results"
except NameError:
    _results_dir = Path(OUTPUT_PATH).parent

_results_dir.mkdir(parents=True, exist_ok=True)
response_checkpoint = _results_dir / (Path(OUTPUT_PATH).stem + ".responses.jsonl")
print(f"Checkpoint path: {response_checkpoint}")

# Resume from checkpoint if it exists
completed: dict = {}  # id -> response
if response_checkpoint.exists():
    with open(response_checkpoint) as f:
        for line in f:
            r = json.loads(line)
            completed[r["id"]] = r["response"]
    print(f"Resumed from checkpoint: {len(completed)} responses already generated")

remaining = [d for d in data if d.get("id") not in completed]
print(f"Generating {len(remaining)} remaining / {len(data)} total...")

for chunk_start in range(0, len(remaining), CHUNK_SIZE):
    chunk = remaining[chunk_start : chunk_start + CHUNK_SIZE]

    prompts = []
    for item in chunk:
        system, user = build_prompt(item["question"], item.get("options"))
        prompt_text = tokenizer.apply_chat_template(
            [{"role": "system", "content": system},
             {"role": "user",   "content": user}],
            tokenize=False,
            add_generation_prompt=True,
        )
        prompts.append(prompt_text)

    with _jupyter_stdout_for_vllm():
        outputs = llm.generate(prompts, sampling_params=sampling_params)

    with open(response_checkpoint, "a") as f:
        for item, out in zip(chunk, outputs):
            response = out.outputs[0].text.strip()
            completed[item["id"]] = response
            f.write(json.dumps({"id": item["id"], "response": response}) + "\n")

    done = len(completed)
    print(f"  {done}/{len(data)}  ({done / len(data) * 100:.1f}%)")

# Ordered responses list matching `data` — consumed by scoring cell below
responses = [completed[d["id"]] for d in data]
print(f"Done. {len(responses)} responses ready.")

Rendering prompts:   0%|          | 0/100 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/100 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Checkpoint path: /content/drive/MyDrive/CSE151B/results/full_public_8k.responses.jsonl
Generating 1126 remaining / 1126 total...


Rendering prompts:   0%|          | 0/100 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/100 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Processed prompts:   0%|          | 0/100 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  1000/1126  (88.8%)


Rendering prompts:   0%|          | 0/26 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/26 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  1100/1126  (97.7%)
  1126/1126  (100.0%)
Done. 1126 responses ready.


## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [7]:
# Load Judger for free-form scoring — same Drive root as data (`DRIVE_PROJECT_ROOT` from the Colab copy cell)
_judger_override = os.environ.get("CSE151B_JUDGER_DIR", "").strip()
try:
    JUDGER_ROOT = _judger_override or str(DRIVE_PROJECT_ROOT)
except NameError:
    JUDGER_ROOT = _judger_override or "."

sys.path.insert(0, os.path.abspath(JUDGER_ROOT))
from judger import Judger
judger = Judger(strict_extract=False)

In [8]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

Scoring: 100%|██████████| 1126/1126 [01:04<00:00, 17.44it/s]

Scoring complete. 1126 results.


## 8. Summary

Print accuracy broken down by question type.

In [9]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :  189 /  375  (50.40%)
  Free-form  :  404 /  751  (53.79%)
  Overall    :  593 / 1126  (52.66%)


In [12]:
import re
from collections import defaultdict

TOPIC_PATTERNS = [
    ("sequences/recurrences", re.compile(r"sequence|recurrence|series|arithmetic|geometric|fibonacci", re.I)),
    ("geometry",              re.compile(r"triangle|circle|angle|polygon|area|perimeter|volume|surface|congruent|similar|parallelogram|trapezoid|chord|radius|diameter", re.I)),
    ("limits",               re.compile(r"\blimit\b|lim_|\\lim", re.I)),
    ("linear algebra",        re.compile(r"matrix|determinant|eigenvalue|eigenvector|vector space|rank|linear transform", re.I)),
    ("derivatives",          re.compile(r"derivative|differentiat|\\frac\{d|d/dx", re.I)),
    ("integration",          re.compile(r"integral|integrat|antiderivative|\\int", re.I)),
    ("probability/stats",    re.compile(r"probability|expected value|variance|distribution|random variable|combinat|permutation", re.I)),
    ("number theory",        re.compile(r"prime|divisib|modulo|congruence|gcd|lcm|diophantine", re.I)),
    ("polynomials/algebra",  re.compile(r"polynomial|quadratic|cubic|root|factor|expand|simplify|equation", re.I)),
]

def classify_topic(question: str) -> str:
    for name, pat in TOPIC_PATTERNS:
        if pat.search(question):
            return name
    return "other"

id_to_q = {d['id']: d['question'] for d in data}

topic_correct  = defaultdict(int)
topic_total    = defaultdict(int)
topic_mcq_c    = defaultdict(int)
topic_mcq_t    = defaultdict(int)

for r in results:
    q = id_to_q.get(r['id'], '')
    t = classify_topic(q)
    topic_total[t]   += 1
    topic_correct[t] += int(r['correct'])
    if r['is_mcq']:
        topic_mcq_t[t] += 1
        topic_mcq_c[t] += int(r['correct'])

rows = sorted(topic_total.keys(), key=lambda t: topic_correct[t] / topic_total[t])

print(f"{'Topic':<25} {'N':>5} {'Acc':>7}  {'MCQ N':>6} {'MCQ Acc':>8}")
print("-" * 60)
for t in rows:
    n         = topic_total[t]
    topic_acc = topic_correct[t] / n * 100
    mn        = topic_mcq_t[t]
    ma        = topic_mcq_c[t] / mn * 100 if mn else float('nan')
    print(f"{t:<25} {n:>5} {topic_acc:>6.1f}%  {mn:>6} {ma:>7.1f}%")
print("-" * 60)
total_acc = sum(r['correct'] for r in results) / len(results) * 100
print(f"{'TOTAL':<25} {len(results):>5} {total_acc:>6.1f}%")

# Save topic breakdown to Drive results folder
try:
    _topic_out = DRIVE_PROJECT_ROOT / "results" / "full_public_8k_topics.json"
except NameError:
    _topic_out = Path(OUTPUT_PATH).parent / "full_public_8k_topics.json"

_topic_data = {
    t: {
        "n": topic_total[t],
        "correct": topic_correct[t],
        "accuracy": round(topic_correct[t] / topic_total[t] * 100, 2),
        "mcq_n": topic_mcq_t[t],
        "mcq_correct": topic_mcq_c[t],
        "mcq_accuracy": round(topic_mcq_c[t] / topic_mcq_t[t] * 100, 2) if topic_mcq_t[t] else None,
    }
    for t in topic_total
}
_topic_out.parent.mkdir(parents=True, exist_ok=True)
with open(_topic_out, 'w') as _f:
    json.dump(_topic_data, _f, indent=2)
print(f"Topic breakdown saved to {_topic_out.resolve()}")

Topic                         N     Acc   MCQ N  MCQ Acc
------------------------------------------------------------
number theory                23   30.4%      21    28.6%
sequences/recurrences        75   34.7%      61    34.4%
geometry                    115   35.7%      39    28.2%
limits                       14   35.7%       7    71.4%
probability/stats            82   50.0%      29    69.0%
linear algebra               23   52.2%      19    63.2%
other                       581   56.1%     102    42.2%
derivatives                  12   58.3%      12    58.3%
polynomials/algebra         146   59.6%      30    76.7%
integration                  55   74.5%      55    74.5%
------------------------------------------------------------
TOTAL                      1126   52.7%
Topic breakdown saved to /content/drive/MyDrive/CSE151B/results/full_public_8k_topics.json


## 9. Save Results

Results are written as newline-delimited JSON. The cell prints the **full path** when it runs.

**Where files go**

- **Google Colab** (after the Drive setup cell that defines `DRIVE_PROJECT_ROOT`): written to **`My Drive/CSE151B/results/`** (filename from `OUTPUT_PATH`, default `starter_results.jsonl`).
- **Local, or Colab without that cell:** `OUTPUT_PATH` relative to the working directory (default `results/starter_results.jsonl`).

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [13]:
SAVE_EVAL = True   # Set to False when running on the private test set

# Colab: write straight to My Drive/CSE151B/results/ (requires Drive setup cell). Else use OUTPUT_PATH.

out_path = DRIVE_PROJECT_ROOT / "results" / Path(OUTPUT_PATH).name


out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path.resolve()}")

Saved 1126 records to /content/drive/MyDrive/CSE151B/results/full_public_8k.jsonl


In [14]:
try:
    from google.colab import runtime
    runtime.unassign()
except ImportError:
    print("Not running in Colab — skipping runtime termination.")